## [控制流](https://docs.langchain.com/oss/python/langgraph/graph-api-control-flow)

### 分支结构 - 扇入

#### 一、静态扇入

按触发条件分为两类：
- “与”触发，上游所有节点全部到达后，才会触发当前节点
- “或”触发，上游任意一个节点到达后，就会触发当前节点

In [ ]:
from typing import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    pass

def a(state: State, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    print(f"节点a: {cur_step}")
    return {}

def b(state: State, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    print(f"节点b: {cur_step}")
    return {}

def b_2(state: State, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    print(f"节点b_2: {cur_step}")
    return {}

def c(state: State, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    print(f"节点c: {cur_step}")
    return {}

builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(b_2)
builder.add_node(c)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge("b", "b_2")
# c在a和b_2节点到达后，分别触发
# builder.add_edge("a", "c")
# builder.add_edge("b_2", "c")
# c在a和b_2节点都到达后，才触发
builder.add_edge(["a", "b_2"], "c")
builder.add_edge("c", END)
graph = builder.compile()
graph.invoke({})


In [ ]:
from IPython.display import display

display(graph)

#### 二、动态扇入 - MapReduce

In [ ]:
import operator
from collections import Counter
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class State(TypedDict):
    lines: list[str]
    words: Annotated[list[str], operator.add]
    frequencies: dict[str, int]

class LineTask(TypedDict):
    text: str

def route(state: State):
    """把每句话分别发送给 count_words 节点"""
    return [
        Send("count_words", {"text": line})
        for line in state["lines"]
    ]

def count_words(task: LineTask):
    words = task["text"].split()
    return {"words": words}

def aggregate(state: State):
    frequencies = dict(Counter(state["words"]))
    return {"frequencies": frequencies}


builder = StateGraph(State)
builder.add_node("count_words", count_words)
builder.add_node("aggregate", aggregate)
builder.add_conditional_edges(START, route)
builder.add_edge("count_words", "aggregate")
builder.add_edge("aggregate", END)
graph = builder.compile()

result = graph.invoke({
    "lines": [
        "hello world",
        "hello langgraph",
        "hello llm",
    ]
})
print(result["frequencies"])